# 05 — Tuning Threshold dari Validation

**Tujuan:** memilih ambang probabilitas yang menyeimbangkan precision dan recall kelas bully, hanya dari validation, lalu menyimpannya untuk evaluasi test.

**Prasyarat:** `models/indobert/val_predictions.csv`.

**Estimasi waktu:** 1 menit.

## Glosarium

- **Threshold**: batas probabilitas; 0,5 bukan angka suci.
- **PR curve**: grafik trade-off precision/recall.
- **F1**: metrik pemilih threshold di sini.
- **Leakage**: memilih threshold dari test membuat laporan terlalu optimistis.

## 1. Muat probabilitas validation

Kita hanya membutuhkan label asli dan probabilitas kelas bully. Tidak ada
training ulang dan tidak ada test set. Notebook ini aman dijalankan berkali-kali.

In [ ]:
import json

import pandas as pd
from sklearn.metrics import precision_recall_curve

from cyberbully.config import LABEL_COLUMN, MODELS_DIR, OUT_DIR
from cyberbully.metrics import binary_metrics, choose_threshold

pred_path = MODELS_DIR / "indobert" / "val_predictions.csv"
df = pd.read_csv(pred_path) if pred_path.exists() else None
print("baris:", len(df) if df is not None else 0)

## 2. Kurva precision-recall

Menurunkan threshold menaikkan recall (lebih banyak bully tertangkap) tetapi
biasanya menurunkan precision (lebih banyak false positive). Kurva ini membantu
melihat pilihan yang masuk akal untuk penggunaan, bukan sekadar mengejar F1.

In [ ]:
if df is not None:
    import matplotlib.pyplot as plt

    precision, recall, _ = precision_recall_curve(df[LABEL_COLUMN], df["probability"])
    fig, ax = plt.subplots(figsize=(6.4, 4))
    ax.plot(recall, precision, color="#0e5f55")
    ax.set_xlabel("recall kelas bully")
    ax.set_ylabel("precision kelas bully")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()
else:
    print("Dilewati: prediksi belum ada.")

## 3. Pilih threshold dan simpan

`choose_threshold` memindai 0,05–0,95 dan memilih F1 bully terbaik. Hasilnya
disimpan ke `out/threshold.json` agar notebook evaluasi dan laporan memakai angka
yang sama. Ini keputusan validation, bukan test.

In [ ]:
if df is not None:
    result = choose_threshold(df[LABEL_COLUMN], df["probability"])
    chosen = float(result["threshold"])
    comparison = {
        "threshold": chosen,
        "validation_metrics": result["metrics"],
        "baseline_threshold_0.5": binary_metrics(df[LABEL_COLUMN], df["probability"], threshold=0.5),
    }
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "threshold.json").write_text(json.dumps(comparison, indent=2) + "\n")
    print("threshold:", round(chosen, 2))
    print(json.dumps(result["metrics"], indent=2))
else:
    print("Dilewati: prediksi belum ada.")

## 4. Bandingkan dengan 0,5

Tabel ini memperlihatkan harga dari keputusan threshold: berapa FN/FP yang
ditukar. Untuk alat bantu moderasi, trade-off ini sebaiknya didiskusikan dengan
pemilik produk dan dicatat di laporan.

In [ ]:
if df is not None:
    rows = []
    for label, value in (("0.5", 0.5), (f"tuned ({chosen:.2f})", chosen)):
        metrics = binary_metrics(df[LABEL_COLUMN], df["probability"], threshold=value)
        rows.append({"threshold": label, **{k: metrics[k] for k in ("precision_bully", "recall_bully", "f1_bully")}})
    print(pd.DataFrame(rows).round(3).to_string(index=False))
else:
    print("Dilewati: prediksi belum ada.")

## Ringkasan

- Threshold dipilih dari validation, bukan test.
- PR curve menunjukkan trade-off precision/recall.
- `out/threshold.json` menjadi sumber angka untuk evaluasi test dan laporan.
- Test set tetap belum disentuh di notebook ini.